# 第二课:向量与矩阵——大模型的原子和积木

讲义:[02-vectors.md](02-vectors.md)。本 notebook 是代码现场:每个 cell 对应讲义一节,顺序执行。

## 1. 向量的基本操作(讲义 §2)

In [1]:
import numpy as np

a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

print('a + b =', a + b)          # 对应位置相加:混合含义
print('2 * b =', 2 * b)          # 数乘:整体缩放
print('norm([3,4]) =', np.linalg.norm([3, 4]))    # 勾股定理 3-4-5
print('norm(a) =', round(np.linalg.norm(a), 4))

a + b = [5 7 9]
2 * b = [ 8 10 12]
norm([3,4]) = 5.0
norm(a) = 3.7417


## 2. 句子 = 矩阵:一摞向量(讲义 §3)

行 = 词,列 = 维度。这是第六课玩具的同款向量。

In [ ]:
words = ['小猫', '它', '可爱']
X = np.array([[1.0, 1.0, 0.0, 0.0],
              [0.9, 1.0, 0.0, 0.0],
              [0.0, 0.0, 1.0, 1.0]])

print('shape =', X.shape)        # [3 个词, 4 维]
print('「它」的向量 =', X[1])

## 3. 矩阵 × 向量:纯 Python 手撕 vs numpy(讲义 §6)

In [ ]:
def matvec(W, x):
    out = []
    for row in W:
        s = 0
        for w_ij, x_j in zip(row, x):
            s += w_ij * x_j
        out.append(s)
    return out

W = [[1, 0, 2],
     [0, 1, 3]]
x = [1, 2, 3]
print('手撕 matvec =', matvec(W, x))          # 期望 [7, 11]
print('numpy  W@x  =', np.array(W) @ np.array(x))

## 4. Embedding 查表 = one-hot 乘法(讲义 §5)

In [ ]:
E = np.array([[1., 0, 0],
              [0., 1, 0],
              [1., 1, 0],
              [0., 0, 1]])      # 4 词 × 3 维

onehot = np.array([0, 0, 1, 0])  # 「id=2」的 one-hot
print('onehot @ E =', onehot @ E)
print('E[2]       =', E[2])
print('相等:', np.array_equal(onehot @ E, E[2]))

# 工程上直接查行:
ids = np.array([2, 0, 3])
print('E[ids] =\n', E[ids])     # 一句话(3 词)→ [3, 3] 矩阵

## 5. 生产账本:参数量与显存(讲义 §7)

In [ ]:
def embedding_bytes(vocab, d, bytes_per_param=4):
    n = vocab * d
    return n, n * bytes_per_param

n, mb = embedding_bytes(50_000, 512, 4)
print(f'50,000 × 512 = {n:,} 参数, float32 = {mb/1024/1024:.1f} MB')

n, gb = embedding_bytes(128_256, 4096, 2)
print(f'128,256 × 4096 = {n:,} 参数, bf16 = {gb/1024**3:.2f} GB')

## 作业现场

- C1:在下面实现纯 Python `matmul(A, B)`,用 X @ X.T 对照 numpy。
- D1/D3:用 `embedding_bytes` 算你作业里的生产题。
- 跑通后回到讲义对答案。

In [ ]:
# TODO(C1): 实现 matmul(A, B) —— 三层 for 循环
# 期望:matmul(X.tolist(), X.T.tolist()) 与 np.round(X @ X.T, 6) 逐元素相等

def matmul(A, B):
    n, k = len(A), len(A[0])
    k2, m = len(B), len(B[0])
    assert k == k2, f'shape 不对齐: {k} != {k2}'   # 生产习惯:先断言接口
    # 你的代码(提示:B 的第 j 列拿出来和 A 的第 i 行点积)
    ...

# 验证
res = None  # TODO: 调用 matmul
ref = X @ X.T
print('numpy 对照:\n', np.round(ref, 3))